# Mixed Precision Quantization: LLaMA 3.2 → `.pte` (ExecuTorch)

| Property | Value |
|----------|-------|
| **Model** | LLaMA 3.2-1B-Instruct (or pruned/distilled variant) |
| **Source** | HuggingFace Hub **or** local directory |
| **Backend** | ExecuTorch XNNPACK (ARM CPU) |
| **Linear Quantization** | `8da4w` — 8-bit dynamic activation + 4-bit weight |
| **Embedding Quantization** | `8w` — 8-bit weight-only |
| **Output** | `.pte` model + `tokenizer.json` + `tokenizer_config.json` |
| **App Compatibility** | ExecuTorch `LlmModule` (handles KV-cache + tokenizer internally) |

## Mixed Precision — What Changes

| Layer Type | Scheme | Bits/Param | What Happens |
|------------|--------|------------|-------------|
| **Linear** (attention, MLP) | `8da4w` | ~4.5 | Weights stored in **4-bit** with group-wise scales; activations quantized to **int8** dynamically at inference |
| **Embedding** (token lookup) | `8w` | 8 | Weights stored in **int8** with per-channel scales |
| **Norms, scales** | FP16 | 16 | Kept at full/half precision for numerical stability |

### Size Estimate (LLaMA 3.2-1B)

| Format | Approximate Size |
|--------|------------------|
| FP16 (baseline) | ~2.5 GB |
| Uniform INT8 | ~1.2 GB |
| **Mixed 8da4w + 8w** | **~0.6–0.8 GB** |

## Workflow

```
Friend prunes / distills LLaMA on VM
       ↓
Safetensors + config stored on VM  —or—  model on HuggingFace Hub
       ↓
This notebook: optimum-cli export executorch (mixed precision)
       ↓
.pte + tokenizer.json + tokenizer_config.json
       ↓
Copy to Android → app loads via LlmModule
```

## How to Run

1. Run **Cell 2** (Install). Wait for completion.
2. **Runtime → Restart session.**
3. Run **Cell 3** (Auth) — set `HF_TOKEN` first.
4. Run **Cell 4** (Config) — set `MODEL_PATH`.
5. Run **Cell 5** (Export) — produces the `.pte` file.
6. Run **Cell 6** (Verify) — lists output files for Android.
7. Optionally run **Cell 7** (Drive save) or **Cell 8** (Inference test).

In [ ]:
# ══ CELL 2: INSTALL DEPENDENCIES ═════════════════════════════════════════════════
# Run once, then Runtime → Restart session.
#
# The nightly CPU index only has torch/torchao/executorch wheels.
# pip's resolver fails when executorch's PyPI deps (expecttest, flatbuffers,
# hypothesis, kgb, etc.) aren't in the nightly index. Fix: install executorch
# with --no-deps and pre-install its runtime dependencies from PyPI first.
# ═══════════════════════════════════════════════════════════════════════

import subprocess, sys, os, shutil, glob

print("=" * 60)
print("  STEP 1: Clean Environment")
print("=" * 60)

# 1. Locate site-packages
sp = subprocess.run(
    [sys.executable, "-c", "import site; print(site.getsitepackages()[0])"],
    capture_output=True, text=True
).stdout.strip()
print(f"\nsite-packages: {sp}")

# 2. Uninstall everything that could conflict
print("\nRemoving existing installations...")
!pip uninstall -y torch torchvision torchaudio torch-xla torchao executorch \
    optimum-executorch coremltools 2>/dev/null || true

# 3. Physical cleanup of leftover directories
patterns = [
    "torch", "torch-*", "torch_*",
    "torchvision*", "torchaudio*", "torch_xla*",
    "torchao*", "executorch*", "~orch*",
]
removed = []
for pat in patterns:
    for path in glob.glob(os.path.join(sp, pat)):
        shutil.rmtree(path, ignore_errors=True)
        removed.append(os.path.basename(path))
if removed:
    print(f"Removed {len(removed)} leftover dirs.")
else:
    print("Clean.")

!pip cache purge 2>/dev/null || true

# 4. Verify torch is gone
r = subprocess.run([sys.executable, "-c", "import torch"], capture_output=True, text=True)
if r.returncode == 0:
    !rm -rf {sp}/torch {sp}/torch-* {sp}/~orch*
    print("WARNING: force-removed extra torch remnants.")
else:
    print("torch fully removed.")

# ── STEP 2: Install in correct order ─────────────────────────────────────────
NIGHTLY = "https://download.pytorch.org/whl/nightly/cpu"

# 2a. Pre-install ALL executorch PyPI deps (the nightly index doesn't have them,
#     which causes pip's resolver to backtrack and fail)
print("\n" + "=" * 60)
print("  STEP 2a: Install executorch PyPI dependencies")
print("=" * 60)
!pip install --no-cache-dir \
    "coremltools==9.0" \
    "mpmath==1.3.0" \
    expecttest flatbuffers hypothesis kgb \
    parameterized "pytest<9.0" pytest-xdist "pytest-rerunfailures==15.1" \
    pytest-json-report pytorch-tokenizers ruamel.yaml tabulate \
    hydra-core omegaconf pandas "scikit-learn>=1.5"

# 2b. Install torch from nightly (CPU build for export — no CUDA needed)
print("\n" + "=" * 60)
print("  STEP 2b: Install torch (nightly CPU)")
print("=" * 60)
!pip install --no-cache-dir --pre torch --index-url {NIGHTLY}

# 2c. Install torchao from nightly (must match torch version)
print("\n" + "=" * 60)
print("  STEP 2c: Install torchao (nightly, matching torch)")
print("=" * 60)
!pip install --no-cache-dir --pre torchao --index-url {NIGHTLY}

# 2d. Install executorch from nightly with --no-deps (all deps already installed
#     above from PyPI; this avoids the resolver trying to find them in nightly index)
print("\n" + "=" * 60)
print("  STEP 2d: Install executorch (nightly, no-deps)")
print("=" * 60)
!pip install --no-cache-dir --no-deps --pre executorch --index-url {NIGHTLY}

# 2e. Install optimum-executorch with --no-deps (its version pins are too strict)
print("\n" + "=" * 60)
print("  STEP 2e: Install optimum-executorch (no-deps)")
print("=" * 60)
!pip install --no-cache-dir --no-deps optimum-executorch

# 2f. Install HF stack — use --no-deps for accelerate to prevent it pulling cuda torch
print("\n" + "=" * 60)
print("  STEP 2f: Install HuggingFace stack")
print("=" * 60)
!pip install --quiet -U transformers tokenizers sentencepiece huggingface_hub optimum
!pip install --quiet --no-deps accelerate

# ── STEP 3: Verify everything ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 3: Verification")
print("=" * 60)
diag = subprocess.run([sys.executable, "-c", """
import torch
print(f"  torch          : {torch.__version__}")
assert 'cpu' in torch.__version__ or 'dev' in torch.__version__, \
    f"WRONG TORCH: {torch.__version__} — Colab's CUDA torch crept back in!"
try:
    import torchao; print(f"  torchao        : {torchao.__version__}")
except Exception as e: print(f"  torchao        : ERROR ({e})")
try:
    import executorch; print(f"  executorch     : {getattr(executorch, '__version__', 'installed')}")
except Exception as e: print(f"  executorch     : ERROR ({e})")
try:
    import optimum.executorch; print(f"  optimum-et     : installed")
except Exception as e: print(f"  optimum-et     : ERROR ({e})")
import transformers
print(f"  transformers   : {transformers.__version__}")

# Verify the critical import chain
try:
    from executorch.extension.llm.custom_ops.custom_ops import custom_sdpa
    print(f"  custom_sdpa    : ✓")
except Exception as e:
    print(f"  custom_sdpa    : ✗ ({e})")

try:
    from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
    print(f"  XnnpackPart    : ✓")
except Exception as e:
    print(f"  XnnpackPart    : ✗ ({e})")

import shutil
cli = shutil.which('optimum-cli')
print(f"  optimum-cli    : {cli if cli else 'NOT FOUND'}")
"""], capture_output=True, text=True)
print(diag.stdout)
if diag.returncode != 0:
    print("⚠️  Diagnostic failed:")
    print(diag.stderr[-800:] if diag.stderr else "(no stderr)")
print("=" * 60)
print("\n✅ Done. Runtime → Restart session → run cells 3 onwards.")

In [ ]:
# ══ CELL 3: HUGGINGFACE AUTHENTICATION ═══════════════════════════════════════
# meta-llama/Llama-3.2-1B-Instruct is a GATED model.
# You must:
#   1. Accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
#   2. Get a HuggingFace token at https://huggingface.co/settings/tokens
#   3. Store it in Colab Secrets (key: HF_TOKEN) or paste below.
# ═══════════════════════════════════════════════════════════════════════

from huggingface_hub import login

HF_TOKEN = ""

# Try Colab Secrets first
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Token loaded from Colab Secrets.")
except Exception:
    pass

# Fallback: paste your token here
if not HF_TOKEN:
    HF_TOKEN = ""  # <-- paste your HuggingFace token here
    if HF_TOKEN:
        print("Using manually provided token.")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is empty.\n"
        "  Option A: Add to Colab Secrets (Key: HF_TOKEN)\n"
        "  Option B: Paste in HF_TOKEN = '...' above\n\n"
        "  NOTE: meta-llama/Llama-3.2-1B-Instruct is GATED.\n"
        "  Accept the license first: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct"
    )

login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to HuggingFace.")

In [ ]:
# ══ CELL 3.5: MOUNT GOOGLE DRIVE ═════════════════════════════════════════════
# Required so the local MODEL_PATH on Drive is accessible.
# ═══════════════════════════════════════════════════════════════════════

import os

try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
        print("Google Drive mounted.")
    else:
        print("Google Drive already mounted.")
except ImportError:
    print("Not running on Colab — skipping Drive mount.")

In [ ]:
# ══ CELL 4: CONFIGURATION ═══════════════════════════════════════════════════
#
# ★ Set MODEL_PATH to your model ★
#
# HuggingFace Hub (base model — use this for now):
#   MODEL_PATH = "meta-llama/Llama-3.2-1B-Instruct"
#
# When your teammate's pruned/KD model is ready:
#   MODEL_PATH = "/content/drive/MyDrive/pruned_llama"   # Google Drive
#   MODEL_PATH = "/content/pruned_llama"                 # uploaded folder
#   MODEL_PATH = "your-org/llama-pruned-distilled"       # custom HF repo
# ═══════════════════════════════════════════════════════════════════════

import os

# ---- Model Source ----
MODEL_PATH = "/content/drive/MyDrive/Navamohan_Model"

# ---- ExecuTorch Backend ----
RECIPE = "xnnpack"        # XNNPACK = optimized for ARM CPU (Android)

# ---- Mixed Precision Quantization ----
QLINEAR    = "8da4w"      # 8-bit dynamic activation + 4-bit weight (Linear layers)
QEMBEDDING = "8w"         # 8-bit weight-only (Embedding layers)

# ---- Output ----
OUTPUT_DIR = "./llama32_1b_mixed_precision"

# ---- Derived ----
IS_LOCAL = os.path.isdir(MODEL_PATH)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("  Configuration")
print("=" * 60)
print(f"  Model       : {MODEL_PATH} ({'local' if IS_LOCAL else 'HuggingFace Hub'})")
print(f"  Recipe      : {RECIPE}")
print(f"  Linear Q    : {QLINEAR}  (8-bit activation, 4-bit weight)")
print(f"  Embedding Q : {QEMBEDDING}  (8-bit weight-only)")
print(f"  Output dir  : {OUTPUT_DIR}")
print("=" * 60)

# Show the exact CLI command that will be executed
print(f"\nCommand that will be executed:\n")
print(f'  optimum-cli export executorch \\')
print(f'      --model "{MODEL_PATH}" \\')
print(f'      --task "text-generation" \\')
print(f'      --recipe "{RECIPE}" \\')
print(f'      --qlinear {QLINEAR} \\')
print(f'      --qembedding {QEMBEDDING} \\')
print(f'      --output_dir="{OUTPUT_DIR}"')

In [ ]:
# ══ CELL 5: EXPORT MODEL → .pte (MIXED PRECISION) ═══════════════════════════
#
# Uses optimum-cli to run the full pipeline:
#   1. Download model from HuggingFace (or load from local path)
#   2. Apply 8da4w quantization to Linear layers
#   3. Apply 8w quantization to Embedding layers
#   4. Export to ExecuTorch with XNNPACK partitioning
#   5. Save .pte file + tokenizer to output directory
#
# The .pte is compatible with ExecuTorch's LlmModule which handles
# KV-cache, tokenization, and autoregressive decoding internally.
# ═══════════════════════════════════════════════════════════════════════

import subprocess, sys, time, os

print("=" * 60)
print("  EXPORTING MODEL → .pte (Mixed Precision)")
print("=" * 60)
print(f"  Model     : {MODEL_PATH}")
print(f"  Linear    : {QLINEAR} (8-bit dynamic activation + 4-bit weight)")
print(f"  Embedding : {QEMBEDDING} (8-bit weight-only)")
print(f"  Backend   : {RECIPE} (ARM CPU optimized)")
print(f"  Output    : {OUTPUT_DIR}")
print("=" * 60)
print("\n  This may take 10-30 min depending on hardware...\n")

# Build the CLI command as a shell string so output streams in real-time
cmd_str = (
    f'optimum-cli export executorch'
    f' --model "{MODEL_PATH}"'
    f' --task "text-generation"'
    f' --recipe "{RECIPE}"'
    f' --qlinear {QLINEAR}'
    f' --qembedding {QEMBEDDING}'
    f' --output_dir="{OUTPUT_DIR}"'
)

print(f"Running:\n  {cmd_str}\n")
print("-" * 60)

# Set HF_TOKEN in environment for the subprocess
env = os.environ.copy()
try:
    env["HF_TOKEN"] = HF_TOKEN
except NameError:
    pass  # Token may already be in env from huggingface-cli login

t0 = time.time()

# Stream stdout/stderr in real-time so we can see actual errors
process = subprocess.Popen(
    cmd_str,
    shell=True,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Print output line by line as it arrives
for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()
elapsed = time.time() - t0

print("-" * 60)
if returncode == 0:
    print(f"\n{'='*60}")
    print(f"  ✅ Export succeeded! ({elapsed/60:.1f} min)")
    print(f"{'='*60}")
else:
    print(f"\n{'='*60}")
    print(f"  ❌ Export FAILED (exit code {returncode})")
    print(f"{'='*60}")
    print("\n  Troubleshooting:")
    print("  • 'Access denied' → Accept LLaMA license on HuggingFace first")
    print("  • 'Out of memory' → Use a machine with ≥32 GB RAM")
    print("  • 'Module not found' → Re-run Cell 2 + restart runtime")
    print("  • 'optimum-cli not found' → pip install optimum-executorch")
    print("\n  The full error output is printed above ↑")

In [ ]:
# ══ CELL 6: VERIFY OUTPUT + COLLECT FILES FOR ANDROID ═══════════════════

import os, shutil

print("=" * 60)
print("  VERIFYING OUTPUT")
print("=" * 60)

# Scan output directory
pte_file = None
tokenizer_file = None
config_file = None

print(f"\nFiles in {OUTPUT_DIR}/:\n")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        if size > 1e9:
            label = f"{size/1e9:.2f} GB"
        elif size > 1e6:
            label = f"{size/1e6:.1f} MB"
        else:
            label = f"{size/1e3:.1f} KB"
        print(f"  {f:45s} {label}")

        if f.endswith(".pte"):
            pte_file = fpath
        elif f == "tokenizer.json":
            tokenizer_file = fpath
        elif f == "tokenizer_config.json":
            config_file = fpath

# .pte validation
if pte_file:
    size_mb = os.path.getsize(pte_file) / (1024 * 1024)
    print(f"\n  .pte file: {os.path.basename(pte_file)} ({size_mb:.1f} MB)")
else:
    print("\n  No .pte file found! Export may have failed.")

# If tokenizer files weren't in the output, download them separately
if not tokenizer_file or not config_file:
    print("\n  Tokenizer files missing from output \u2014 downloading separately...")
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        token=HF_TOKEN if not IS_LOCAL else None,
    )
    tok.save_pretrained(OUTPUT_DIR)
    tokenizer_file = os.path.join(OUTPUT_DIR, "tokenizer.json")
    config_file = os.path.join(OUTPUT_DIR, "tokenizer_config.json")
    print("  Tokenizer saved.")

# Summary for Android deployment
print(f"\n{'='*60}")
print(f"  FILES FOR ANDROID DEPLOYMENT")
print(f"{'='*60}")
android_files = []
for label, path in [
    ("Model (.pte)", pte_file),
    ("Tokenizer", tokenizer_file),
    ("Tokenizer Config", config_file),
]:
    if path and os.path.exists(path):
        size = os.path.getsize(path)
        unit = f"{size/1e6:.1f} MB" if size > 1e6 else f"{size/1e3:.1f} KB"
        print(f"  {label:20s}: {os.path.basename(path):30s} ({unit})")
        android_files.append(path)
    else:
        print(f"  {label:20s}: NOT FOUND")

print(f"\n  Copy these {len(android_files)} files to your Android device.")
print(f"  The app's LlmModule will auto-detect the tokenizer and chat template.")
print(f"\n  Chat template: LLaMA 3 format")
print(f"    <|begin_of_text|><|start_header_id|>system<|end_header_id|>...")
print(f"    (auto-detected from tokenizer_config.json)")
print(f"{'='*60}")

In [ ]:
# ══ CELL 7: SAVE TO GOOGLE DRIVE (OPTIONAL) ══════════════════════════

import os, shutil

DRIVE_DIR = "/content/drive/MyDrive/ExecuTorch_Models/llama32_1b"

try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

    # Copy all files from output directory to Drive
    for f in os.listdir(OUTPUT_DIR):
        src = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(src):
            dst = os.path.join(DRIVE_DIR, f)
            shutil.copy2(src, dst)
            size = os.path.getsize(dst)
            print(f"  Saved: {f} ({size/1e6:.1f} MB)")

    print(f"\nAll files saved to: {DRIVE_DIR}")
except ImportError:
    print("Not running on Colab \u2014 skipping Google Drive save.")
    print(f"Files are available locally in: {os.path.abspath(OUTPUT_DIR)}")

## Cell 8 \u2014 Inference Test (Optional)

Load the `.pte` produced by Cell 5 and run a quick prompt in Colab.  
This uses ExecuTorch\u2019s Python LLM runner.  
**Note:** Expect slow performance on Colab CPU (\u223c1\u20133 tok/s without KV cache).

In [ ]:
# ══ CELL 8: INFERENCE TEST (OPTIONAL) ═════════════════════════════════
#
# Quick sanity check that the .pte loads and produces output.
# Real performance testing should be done on the Android device.
# ═══════════════════════════════════════════════════════════════════════

import os, gc, time, contextlib
import torch

# Find the .pte file
PTE_FILE = None
for f in os.listdir(OUTPUT_DIR):
    if f.endswith(".pte"):
        PTE_FILE = os.path.join(OUTPUT_DIR, f)
        break

if not PTE_FILE:
    raise FileNotFoundError(f"No .pte file found in {OUTPUT_DIR}")

print(f"PTE file: {PTE_FILE} ({os.path.getsize(PTE_FILE)/1e6:.1f} MB)")

# \u2014 Suppress C++ stderr noise from XNNPACK init \u2014
@contextlib.contextmanager
def _quiet():
    try:
        devnull = os.open(os.devnull, os.O_WRONLY)
        saved   = os.dup(2)
        os.dup2(devnull, 2)
        os.close(devnull)
        try:
            yield
        finally:
            os.dup2(saved, 2)
            os.close(saved)
    except Exception:
        yield

# \u2014 Load ExecuTorch runtime \u2014
try:
    from executorch.extension.pybindings.portable_lib import _load_for_executorch
except ImportError:
    raise ImportError(
        "ExecuTorch Python bindings not found.\n"
        "  Re-run Cell 2, restart runtime, then try again."
    )

gc.collect()
print("Loading .pte into ExecuTorch runtime...")
with _quiet():
    et_module = _load_for_executorch(PTE_FILE)
print("ExecuTorch module loaded.")

# \u2014 Load tokenizer \u2014
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
print(f"Tokenizer loaded (vocab={tokenizer.vocab_size})")

# \u2014 Generate \u2014
MAX_NEW_TOK = 30  # keep short for CPU

def generate(prompt: str, max_new_tokens: int = MAX_NEW_TOK) -> str:
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    attn_mask = torch.ones_like(input_ids)
    generated = []

    print(f"  Prompt: {prompt!r}")
    print(f"  Response: ", end="", flush=True)

    t0 = time.time()
    for _ in range(max_new_tokens):
        with _quiet():
            outputs = et_module.forward((input_ids, attn_mask))
        logits = outputs[0]
        next_tok = int(logits[0, -1, :].argmax())

        if next_tok == tokenizer.eos_token_id:
            break

        generated.append(next_tok)
        word = tokenizer.decode([next_tok], skip_special_tokens=True)
        print(word, end="", flush=True)

        new_id = torch.tensor([[next_tok]], dtype=torch.long)
        input_ids = torch.cat([input_ids, new_id], dim=1)
        attn_mask = torch.ones_like(input_ids)

    elapsed = time.time() - t0
    toks = len(generated)
    print(f"\n  [{toks} tokens | {elapsed:.1f}s | {toks/max(elapsed,0.1):.1f} tok/s]")
    return tokenizer.decode(generated, skip_special_tokens=True)

print("\n" + "=" * 60)
print("  LLaMA 3.2-1B Mixed Precision \u2014 ExecuTorch on Colab CPU")
print("=" * 60)

generate("What is the capital of France?")
print("-" * 60)
generate("Explain quantization in one sentence.")
print("-" * 60)

print("\nDone. Call generate('your question') for custom prompts.")